# BRCA internal val — UNI2 A8 (gate + balanced L12)

Xenium BRCA. A8 `best_mlp_gpu.pt` is the fold-0 checkpoint (`hce_group_fold_0.pt`, three-tier 2.6139). Its held-out BRCA section is `rep1`. `rep2` is fold 1 and is not shown.

Checkpoint: `data/PanCancer/result_uni2_A8_l12_balanced_paired_v2_seed42/cross_dataset_cv/A8_l12_balanced/best_mlp_gpu.pt`  
k=8 mean, learned spatial gate, balanced L12. This notebook only reads predictions. The A8 deployment tree is separate from the published A2 tree.

```bash
ONLY=BRCA CUDA_DEVICE=0 bash code/Internal_val/demo_uni2_A8.sh predict
```


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

REPO = Path("/home/lingyu/ssd2/Python/Hist2Pheno")
if not (REPO / "code").is_dir():
    REPO = Path("/nobackup2/users/lingyu/Python/Hist2Pheno")
sys.path.insert(0, str(REPO / "code" / "Internal_val"))

import internal_val as iv

# A8 best_mlp is fold 0. Other Internal_val notebooks stay on the published A2 paths.
iv.FOLD = 0
iv.PRED_ROOT = REPO / "data" / "PanCancer" / "pred_uni2_A8_l12_balanced_paired_v2_seed42_deployment"
iv.CKPT = (
    REPO
    / "data"
    / "PanCancer"
    / "result_uni2_A8_l12_balanced_paired_v2_seed42"
    / "cross_dataset_cv"
    / "A8_l12_balanced"
    / "best_mlp_gpu.pt"
)
iv.FOLD_PLAN = iv.CKPT.parents[2] / "fold_plan.csv"
iv.METRICS = iv.PRED_ROOT / "metrics_per_section.csv"
iv.FIG_DIR = iv.PRED_ROOT / "internal_val_figures"

DATASET = "BRCA"
SAVE_FIGURES = False
fold = iv.fold0_table(DATASET)
viz = iv.viz_samples(DATASET)
cov = iv.coverage(DATASET)
print(f"{DATASET} fold {iv.FOLD}: {len(fold)} sections; drawing {len(viz)}")
display(cov)
met = iv.metrics_fold0(DATASET)
if not met.empty:
    cols = [c for c in ("sample", "n_matched", "macro_auc_L1_cells", "macro_auc_L12_cells", "macro_auc_L2_cells") if c in met]
    display(met[cols])
print("viz", viz)


In [ ]:
display(Markdown("## Existing 4×3 panels (GT / Hist2Pheno / ROC / StarDist-all)"))
missing = []
for sample in viz:
    panel = iv.pred_paths(DATASET, sample)["panel"]
    if panel.is_file():
        display(Markdown(f"### {sample}"))
        display(Image(filename=str(panel)))
    else:
        missing.append(sample)
        print("missing panel", sample)
if missing:
    print("A8 deployment did not save 4x3 panel PNGs. GT versus prediction is drawn from matched_pred.h5ad.")


In [ ]:
display(Markdown("## GT vs pretrained L2 (matched nuclei)"))
for sample in viz:
    pred = iv.pred_paths(DATASET, sample)["pred"]
    if not pred.is_file():
        print("skip", sample, "(no matched_pred.h5ad)")
        continue
    fig = iv.plot_spatial_gt_pred(DATASET, sample, "L2", save=SAVE_FIGURES)
    display(fig)
    plt.close(fig)
    fig = iv.plot_composition(DATASET, sample, "L2", save=SAVE_FIGURES)
    display(fig)
    plt.close(fig)
